# Taller de Procesamiento de Datos con Apache Spark

**Estudiante:** Isabella Achury Dussan  
**Programa:** Ciencia de Datos  
**Semestre:** VI semestre  
**Asignatura:** Procesamiento de Datos  
**Docente:** Jhon Corredor  
**Universidad:** Pontificia Universidad Javeriana 
**Fecha:** 21 de septiembre de 2026  

---

## Objetivo

Desarrollar un proceso básico de procesamiento y análisis de datos utilizando Apache Spark y PySpark, empleando el conjunto de datos Iris para realizar la preparación de los datos, transformación de características, entrenamiento de modelos de clasificación y evaluación de sus resultados.

In [1]:
import os

# Evitar que se utilice el Spark 4.2.0 del clúster
os.environ.pop("SPARK_HOME", None)
os.environ.pop("SPARK_CONF_DIR", None)
os.environ.pop("PYSPARK_SUBMIT_ARGS", None)

# Usar Python 3.9
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"

print("SPARK_HOME =", os.environ.get("SPARK_HOME"))
print("SPARK_CONF_DIR =", os.environ.get("SPARK_CONF_DIR"))
print("PYSPARK_PYTHON =", os.environ.get("PYSPARK_PYTHON"))

SPARK_HOME = None
SPARK_CONF_DIR = None
PYSPARK_PYTHON = /usr/bin/python3


In [2]:
import pyspark

print("PySpark =", pyspark.__version__)
print("Ruta =", pyspark.__file__)

PySpark = 4.0.4
Ruta = /home/estudiante/.local/lib/python3.9/site-packages/pyspark/__init__.py


## 1. Configuración de Spark

En esta sección se crea la sesión de Apache Spark mediante `SparkSession`, que permite establecer el entorno necesario para ejecutar las operaciones de procesamiento de datos. La ejecución se realiza en modo local utilizando los recursos disponibles en el equipo.

In [3]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Apache Spark Beginner Tutorial")
         .config("spark.executor.memory", "1G")
         .getOrCreate())

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 21:16:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.4
Master: local[*]


## Importing Libraries

## 2. Importación de librerías

En esta sección se importan las librerías necesarias para desarrollar el ejercicio de procesamiento y clasificación de datos utilizando PySpark. Se utilizan herramientas para la manipulación de datos, construcción de modelos de aprendizaje automático y evaluación de los resultados.

In [4]:
#Generic Libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

#Apache Spark Libraries
import pyspark
from pyspark.sql import SparkSession

#Apache Spark ML CLassifier Libraries
from pyspark.ml.classification import DecisionTreeClassifier,RandomForestClassifier,NaiveBayes

#Apache Spark Evaluation Library
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Apache Spark Features libraries
from pyspark.ml.feature import StandardScaler,StringIndexer, VectorAssembler, VectorIndexer, OneHotEncoder

#Apache Spark Pipelin Library
from pyspark.ml import Pipeline

# Apache Spark `DenseVector`
from pyspark.ml.linalg import DenseVector

#Data Split Libraries
import sklearn
from sklearn.model_selection import train_test_split


#Tabulating Data
from tabulate import tabulate

#Garbage
import gc

## Build Spark Session

In [5]:
#Building Spark Session
spark = (SparkSession.builder
                  .appName('Apache Spark Beginner Tutorial')
                  .config("spark.executor.memory", "1G")
                  .config("spark.executor.cores","4")
                  .getOrCreate())

26/09/21 21:16:19 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
spark.sparkContext.setLogLevel('INFO')

In [7]:
spark.version

'4.0.4'

## Data Load

## 3. Carga del conjunto de datos

En esta sección se carga el conjunto de datos `Iris.csv` utilizando un DataFrame de Apache Spark. Se utiliza la primera fila del archivo como encabezado y se habilita la inferencia automática de los tipos de datos de las diferentes columnas.

In [8]:
url = 'Iris.csv'

data = spark.read.format("csv") \
       .option("header", "true") \
       .option("inferSchema","true")\
       .load(url) 

data.cache() #for faster re-use

DataFrame[Id: int, SepalLengthCm: double, SepalWidthCm: double, PetalLengthCm: double, PetalWidthCm: double, Species: string]

## Data Exploration & Preparation

## 4. Exploración de los datos

En esta etapa se realiza una exploración inicial del conjunto de datos para conocer su tamaño, estructura y contenido. Se consulta la cantidad de registros, el esquema del DataFrame, una muestra de los datos, la distribución de las especies y estadísticas descriptivas de las variables numéricas.

Estas operaciones permiten obtener una visión general de los datos antes de realizar su transformación y preparación para los modelos de clasificación.

In [9]:
#Total records 
data.count()

150

In [10]:
#Data Type
data.printSchema()

root
 |-- Id: integer (nullable = true)
 |-- SepalLengthCm: double (nullable = true)
 |-- SepalWidthCm: double (nullable = true)
 |-- PetalLengthCm: double (nullable = true)
 |-- PetalWidthCm: double (nullable = true)
 |-- Species: string (nullable = true)



In [11]:
#Display records
data.show(5)

+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 5 rows


In [12]:
#Records per Species
data.groupBy('species').count().show()

+---------------+-----+
|        species|count|
+---------------+-----+
| Iris-virginica|   50|
|    Iris-setosa|   50|
|Iris-versicolor|   50|
+---------------+-----+



In [13]:
#Dataset Summary Stats
data.describe().show()

26/09/21 21:16:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+-------------------+------------------+------------------+--------------+
|summary|                Id|     SepalLengthCm|       SepalWidthCm|     PetalLengthCm|      PetalWidthCm|       Species|
+-------+------------------+------------------+-------------------+------------------+------------------+--------------+
|  count|               150|               150|                150|               150|               150|           150|
|   mean|              75.5| 5.843333333333335| 3.0540000000000007|3.7586666666666693|1.1986666666666672|          NULL|
| stddev|43.445367992456916|0.8280661279778637|0.43359431136217375| 1.764420419952262|0.7631607417008414|          NULL|
|    min|                 1|               4.3|                2.0|               1.0|               0.1|   Iris-setosa|
|    max|               150|               7.9|                4.4|               6.9|               2.5|Iris-virginica|
+-------+------------------+----

Inorder for our model to make predictions the Species aka Label column should be a numerical value (models don't like string!). To achieve this we shall use String Indexing on the Species columns

## 5. Transformación de la variable objetivo

La columna `Species`, que contiene las especies de las flores, es una variable categórica. Para utilizarla en los modelos de clasificación, se transforma a una representación numérica mediante `StringIndexer`. El resultado se almacena en la columna `species_indx`.

In [14]:
#String Indexing the Species column
SIndexer = StringIndexer(inputCol='species', outputCol='species_indx')
data = SIndexer.fit(data).transform(data)

#Inspect the dataset
data.show(5)


+---+-------------+------------+-------------+------------+-----------+------------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|species_indx|
+---+-------------+------------+-------------+------------+-----------+------------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|         0.0|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|         0.0|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|         0.0|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|         0.0|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|         0.0|
+---+-------------+------------+-------------+------------+-----------+------------+
only showing top 5 rows


## Feature Engineering

The Spark model needs two columns: “label” and “features” and we are not going to do much feature engineering because we want to focus on the mechanics of training the model in Spark. 

So, creating a seperate dataframe with re-ordered columns, then defining an input data using Dense Vector. A Dense Vector is a local vector that is backed by a double array that represents its entry values. In other words, it's used to store arrays of values for use in PySpark.


## 6. Preparación de las características

En esta etapa se seleccionan las variables numéricas que contienen las características de las flores: longitud y ancho del sépalo y longitud y ancho del pétalo.

Estas variables se utilizan como características de entrada (`features`) para los modelos de clasificación, mientras que `species_indx` representa la variable objetivo (`label`).

In [15]:
#creating a seperate dataframe with re-ordered columns
df = data.select("species_indx", "SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm")

# Inspect the dataframe
df.show(5)

+------------+-------------+------------+-------------+------------+
|species_indx|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|
+------------+-------------+------------+-------------+------------+
|         0.0|          5.1|         3.5|          1.4|         0.2|
|         0.0|          4.9|         3.0|          1.4|         0.2|
|         0.0|          4.7|         3.2|          1.3|         0.2|
|         0.0|          4.6|         3.1|          1.5|         0.2|
|         0.0|          5.0|         3.6|          1.4|         0.2|
+------------+-------------+------------+-------------+------------+
only showing top 5 rows


**Note:** Observe that the species column which is our label (aka Target) is now at beginning of the dataframe

In [16]:
# Define the `input_data` as Dense Vector
input_data = df.rdd.map(lambda x: (x[0], DenseVector(x[1:])))

**Note:** Observe the definition of the Dense Vector. So,when we create a new indexed dataframe(below) the machine understands that the first column is a Label (Target) and the remaining columns are Features.

In [17]:
# Creating a new Indexed Dataframe
df_indx = spark.createDataFrame(input_data, ["label", "features"])

In [18]:
#view the indexed dataframe
df_indx.show(5)

+-----+-----------------+
|label|         features|
+-----+-----------------+
|  0.0|[5.1,3.5,1.4,0.2]|
|  0.0|[4.9,3.0,1.4,0.2]|
|  0.0|[4.7,3.2,1.3,0.2]|
|  0.0|[4.6,3.1,1.5,0.2]|
|  0.0|[5.0,3.6,1.4,0.2]|
+-----+-----------------+
only showing top 5 rows


## Data Scaling

This is also known as Feature Scaling. It is a method of normalizing the features of the data. Scaling can make a difference between a weak machine learning model and a better one. 

In this tutorial we will use a Standard Scaler to scale our feature data. Apache Spark has a Standard Scaler library to do the job.

## 7. Escalamiento de las características

Se aplica `StandardScaler` a las características para estandarizar sus valores. Este proceso permite transformar las variables para que tengan una escala comparable antes de utilizarlas en los modelos de clasificación.

In [19]:
#Initialize Standard Scaler
stdScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

#Fit the Standard Scaler to the indexed Dataframe
scaler = stdScaler.fit(df_indx)

#Transform the dataframe
df_scaled =scaler.transform(df_indx)

In [20]:
#Viewing the Scaled Data
df_scaled.show(5)

+-----+-----------------+--------------------+
|label|         features|     features_scaled|
+-----+-----------------+--------------------+
|  0.0|[5.1,3.5,1.4,0.2]|[6.15892840883878...|
|  0.0|[4.9,3.0,1.4,0.2]|[5.9174018045706,...|
|  0.0|[4.7,3.2,1.3,0.2]|[5.67587520030241...|
|  0.0|[4.6,3.1,1.5,0.2]|[5.55511189816831...|
|  0.0|[5.0,3.6,1.4,0.2]|[6.03816510670469...|
+-----+-----------------+--------------------+
only showing top 5 rows


In [21]:
#Dropping the Features column
df_scaled = df_scaled.drop("features")

## Data Split

Just like always, before building a model we shall split our scaled dataset into training & test sets. 
Training Dataset = 90%
Test Dataset = 10%

## 8. División del conjunto de datos

El conjunto de datos se divide en dos grupos: uno destinado al entrenamiento de los modelos y otro utilizado para evaluar su desempeño.

Se utiliza una proporción de 90 % para entrenamiento y 10 % para prueba, utilizando una semilla (`seed`) para controlar la aleatoriedad de la división.

In [22]:
train_data, test_data = df_scaled.randomSplit([0.9, 0.1], seed = 12345)

In [23]:
#Inspect Training Data
train_data.show(5)

[Stage 27:>                                                         (0 + 1) / 1]

+-----+--------------------+
|label|     features_scaled|
+-----+--------------------+
|  0.0|[5.19282199176603...|
|  0.0|[5.31358529390013...|
|  0.0|[5.31358529390013...|
|  0.0|[5.31358529390013...|
|  0.0|[5.43434859603422...|
+-----+--------------------+
only showing top 5 rows


## Build, Train & Evaluate Model

In this step we will create multiple models, train them on our scaled dataset and then compare their accuracy.

## 9. Entrenamiento del modelo Decision Tree

En esta etapa se entrena un modelo de árbol de decisión utilizando el conjunto de entrenamiento. El modelo aprende patrones a partir de las características de las flores con el objetivo de predecir la especie correspondiente.

In [24]:
model = ['Decision Tree','Random Forest','Naive Bayes']
model_results = []

In [25]:
# -- Decision Tree Classifier --

dtc = DecisionTreeClassifier(labelCol="label", featuresCol="features_scaled")          #instantiate the model
dtc_model = dtc.fit(train_data)                                                        #train the model
dtc_pred = dtc_model.transform(test_data)                                              #model predictions

#Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
dtc_acc = evaluator.evaluate(dtc_pred)
#print("Decision Tree Classifier Accuracy =", '{:.2%}'.format(dtc_acc))
model_results.extend([[model[0],'{:.2%}'.format(dtc_acc)]])                               #appending to list
    

## 10. Entrenamiento del modelo Random Forest

Se entrena un modelo Random Forest utilizando el conjunto de entrenamiento. Este algoritmo combina múltiples árboles de decisión para realizar las predicciones y posteriormente se evalúa su desempeño sobre el conjunto de prueba.

In [26]:
# -- Random Forest Classifier --

rfc = RandomForestClassifier(labelCol="label", featuresCol="features_scaled", numTrees=10)          #instantiate the model
rfc_model = rfc.fit(train_data)                                                                     #train the model
rfc_pred = rfc_model.transform(test_data)                                                           #model predictions

#Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
rfc_acc = evaluator.evaluate(rfc_pred)
#print("Random Forest Classifier Accuracy =", '{:.2%}'.format(rfc_acc))
model_results.extend([[model[1],'{:.2%}'.format(rfc_acc)]])                                            #appending to list

## 11. Entrenamiento del modelo Naive Bayes

Se entrena un modelo Naive Bayes utilizando las características preparadas anteriormente. El modelo se utiliza para clasificar las observaciones según la especie de la flor y posteriormente se evalúa su precisión.

In [27]:
# -- Naive Bayes Classifier --

nbc = NaiveBayes(smoothing=1.0,modelType="multinomial", labelCol="label",featuresCol="features_scaled")    #instantiate the model
nbc_model = nbc.fit(train_data)                                                                          #train the model
nbc_pred = nbc_model.transform(test_data)                                                                #model predictions

#Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
nbc_acc = evaluator.evaluate(nbc_pred)
#print("Naive Bayes Accuracy =", '{:.2%}'.format(nbc_acc))
model_results.extend([[model[2],'{:.2%}'.format(nbc_acc)]])                                            #appending to list

26/09/21 21:16:39 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [28]:
#freeing memory
gc.collect()

492

Tabulating the results.

## 12. Evaluación de los modelos

Para evaluar el desempeño de los modelos se utiliza la métrica `accuracy`, que representa la proporción de predicciones correctas realizadas sobre el conjunto de prueba.

La precisión obtenida permite comparar el comportamiento de los tres modelos de clasificación utilizados en el ejercicio.

In [29]:
print (tabulate(model_results, headers=["Classifier Models", "Accuracy"]))

Classifier Models    Accuracy
-------------------  ----------
Decision Tree        90.91%
Random Forest        100.00%
Naive Bayes          100.00%


## Conclusión

A partir de los resultados obtenidos, se observa que los tres modelos presentaron un buen desempeño en la clasificación de las especies del conjunto de datos Iris. El modelo Decision Tree alcanzó una precisión del 90,91 %, mientras que Random Forest y Naive Bayes alcanzaron una precisión del 100 % sobre el conjunto de prueba utilizado.

Estos resultados muestran que, para la ejecución realizada, los modelos Random Forest y Naive Bayes clasificaron correctamente todas las observaciones pertenecientes al conjunto de prueba, mientras que Decision Tree presentó un pequeño margen de error.

El desarrollo del taller permitió aplicar un proceso completo de procesamiento y análisis de datos utilizando Apache Spark y PySpark, incluyendo la carga y exploración del conjunto de datos, transformación de variables, preparación de características, escalamiento, división de los datos, entrenamiento de modelos y evaluación mediante la métrica de accuracy.